# financial-rag-aws — IR training (Colab)

Mines training pairs, finetunes **bge-small**, trains a **cross-encoder reranker**, and reads
the finetune lift with an in-memory retrieval eval. Runs on a free Colab GPU; no AWS is needed
for training — artifacts are saved locally and uploaded to S3 later.


## 1. Environment


In [ ]:
# Set runtime to GPU: Runtime -> Change runtime type -> T4 GPU
REPO = 'https://github.com/Hydaspex/financial-rag-aws'  # adjust to your fork
!git clone -q $REPO
%cd financial-rag-aws
!pip -q install -e '.[dev]'


In [ ]:
import getpass, os
# OpenRouter is only needed for synthetic query generation in step 3.
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OPENROUTER_API_KEY: ')
os.environ['PAIRGEN_MODEL'] = 'meta-llama/llama-3.3-70b-instruct'


## 2. Corpus
Provide a corpus JSONL of chunk records (`{id, text, metadata}`). Either commit a small
`data/corpus.jsonl`, or pull it from S3 once the data lake is up:

```python
# from frag.aws.s3_store import download_jsonl
# recs = download_jsonl('frag-datalake-452575447847', 'corpus/corpus.jsonl')
```


## 3. Mine training pairs (synthetic queries + hard negatives)


In [ ]:
!python scripts/mine_pairs.py --corpus data/corpus.jsonl --out data/pairs.jsonl \
    --queries-per-passage 2 --n-negatives 4 --max-passages 500


## 4. Finetune the embedding model


In [ ]:
!python scripts/finetune_embedding.py --pairs data/pairs.jsonl --out artifacts/bge-ft --epochs 1


## 5. Train the cross-encoder reranker


In [ ]:
!python scripts/train_reranker.py --pairs data/pairs.jsonl --out artifacts/reranker --epochs 1


## 6. Read the finetune lift (in-memory eval, no AWS)


In [ ]:
print('--- base bge-small ---')
!python scripts/eval_local.py --corpus data/corpus.jsonl --model BAAI/bge-small-en-v1.5
print('--- finetuned ---')
!python scripts/eval_local.py --corpus data/corpus.jsonl --model artifacts/bge-ft


## 7. Upload artifacts to S3 (when the data lake is up)
```python
# import boto3; s3 = boto3.Session(profile_name='ninefin').client('s3')
# for f in ['bge-ft','reranker']:  # tar and put under artifacts/
#     ...
```
